# 2 序列模型 - 2.1 理论计算题
## 一、基础统计
### 1. 拆分一阶转移对
序列字符顺序：$a,b,a,b,c$
相邻转移二元组（前驱→后继）：
$(a,b)、(b,a)、(a,b)、(b,c)$

### 2. 针对前驱字符 `b` 统计计数
- $\text{count}(b \to a) = 1$
- $\text{count}(b \to c) = 1$
- $\text{count}(b \to b) = 0$
- $\text{count}(b)$：`b`作为前驱总出现次数 $= 2$

### 3. 拉普拉斯平滑公式
$$
p(y | x) = \frac{\text{count}(x \to y) + 1}{\text{count}(x) + |V|}
$$

## 二、分步计算
### 1. 计算 $p(\text{'a'} | \text{'b'})$
$$
\begin{align*}
p(a | b) &= \frac{\text{count}(b\to a)+1}{\text{count}(b)+|V|} \\
&= \frac{1+1}{2+3} \\
&= \frac{2}{5} = 0.4
\end{align*}
$$

### 2. 计算 $p(\text{'c'} | \text{'b'})$
$$
\begin{align*}
p(c | b) &= \frac{\text{count}(b\to c)+1}{\text{count}(b)+|V|} \\
&= \frac{1+1}{2+3} \\
&= \frac{2}{5} = 0.4
\end{align*}
$$

## 三、归一化校验（验证）
`b` 全部转移概率：
$$
p(a|b)=\frac{2}{5},\quad p(b|b)=\frac{0+1}{5}=\frac{1}{5},\quad p(c|b)=\frac{2}{5}
$$
求和：$\displaystyle \frac{2}{5}+\frac{1}{5}+\frac{2}{5} = 1$，满足概率归一化要求。

## 四、最终结果
1. $p(\text{'a'} | \text{'b'}) = \boldsymbol{\dfrac{2}{5}}$
2. $p(\text{'c'} | \text{'b'}) = \boldsymbol{\dfrac{2}{5}}$

In [1]:
import string
from collections import Counter

def preprocess_text(text, n):
    # 1. 转小写，去除所有标点，仅保留字母和空格
    text_lower = text.lower()
    punc_table = str.maketrans('', '', string.punctuation)
    text_clean = text_lower.translate(punc_table)
    
    # 2. 按空格分词
    word_list = text_clean.split()
    
    # 3. 按词频降序构建词汇表，ID从0开始
    word_freq = Counter(word_list)
    # 先按频次从大到小，频次相同按单词排序
    sorted_words = sorted(word_freq.keys(), key=lambda x: (-word_freq[x], x))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 4. 滑动窗口生成特征与标签
    features = []
    labels = []
    total_len = len(word_list)
    max_start = total_len - n
    if max_start < 0:
        return vocab, (features, labels)
    
    # 遍历所有窗口起始位置
    for start in range(max_start + 1):
        window = word_list[start : start + n]
        features.append(window)
        # 判断是否存在下一个词
        next_pos = start + n
        if next_pos < total_len:
            labels.append(word_list[next_pos])
        else:
            labels.append(None)
    
    return vocab, (features, labels)

# 测试用例
if __name__ == "__main__":
    test_str = "The time machine"
    vocab, (feat, lab) = preprocess_text(test_str, n=2)
    print("词汇表：", vocab)
    print("特征：", feat)
    print("标签：", lab)

词汇表： {'machine': 0, 'the': 1, 'time': 2}
特征： [['the', 'time'], ['time', 'machine']]
标签： ['machine', None]


# 3 循环神经网络 3.1 计算题
## 已知公式
1. 隐状态：$h_t = W_{hh}h_{t-1}+W_{hx}x_t$
2. 输出：$o_t=W_{oh}h_t$
3. 损失：$L=\frac12\sum_{t=1}^T(o_t-y_t)^2$

## 步骤1：求单步梯度
- 损失对输出：$\displaystyle\frac{\partial L}{\partial o_t}=o_t-y_t$
- 输出对隐状态：$\displaystyle\frac{\partial o_t}{\partial h_t}=W_{oh}^\top$
- 令 $\delta_t=\frac{\partial L}{\partial h_t}$，则 $\delta_t = W_{oh}^\top(o_t-y_t)$

## 步骤2：BPTT反向递推
隐状态传递导数：$\displaystyle\frac{\partial h_t}{\partial h_{t-1}}=W_{hh}$
反向递推：$\delta_{k}= (W_{hh}^\top)^{T-k}\delta_T$

## 步骤3：损失对$W_{hh}$梯度
每个时刻k贡献梯度：$\delta_k h_{k-1}^\top$
累加全部时间步：
$$
\boldsymbol{\frac{\partial L}{\partial W_{hh}}=\sum_{k=1}^T \delta_k h_{k-1}^\top
=\sum_{k=1}^T \big(W_{hh}^\top\big)^{T-k} W_{oh}^\top(o_T-y_T) h_{k-1}^\top}
$$

## 步骤4：梯度消失/爆炸条件
设 $W_{hh}$ 谱半径（最大特征值模）$\rho$：
1. 梯度爆炸：$\boldsymbol{\rho(W_{hh})>1}$，矩阵幂次指数放大梯度
2. 梯度消失：$\boldsymbol{\rho(W_{hh})<1}$，矩阵幂次指数衰减梯度

In [2]:
import numpy as np

def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN单步前向传播
    参数:
        x_t: (batch_size, input_size)
        h_prev: (batch_size, hidden_size)
        W_hx: (hidden_size, input_size)
        W_hh: (hidden_size, hidden_size)
        b_h: (1, hidden_size)
    返回:
        h_t: 当前隐状态 (batch_size, hidden_size)
        cache: 缓存前向中间变量，用于反向传播
    """
    # 线性变换 z_t = x_t @ W_hx.T + h_prev @ W_hh.T + b_h
    z_t = np.dot(x_t, W_hx.T) + np.dot(h_prev, W_hh.T) + b_h
    h_t = np.tanh(z_t)
    cache = (x_t, h_prev, W_hx, W_hh, b_h, z_t)
    return h_t, cache


def rnn_backward(dh_next, cache):
    """
    RNN单步反向传播，计算各参数梯度
    参数:
        dh_next: 上游梯度 dL/dh_t (batch_size, hidden_size)
        cache: 前向传播保存的中间变量
    返回:
        dx_t, dh_prev, dW_hx, dW_hh, db_h
    """
    x_t, h_prev, W_hx, W_hh, b_h, z_t = cache
    batch_size = x_t.shape[0]
    
    # tanh导数: dL/dz_t = dh_next * (1 - tanh(z_t)^2)
    dz_t = dh_next * (1 - np.square(np.tanh(z_t)))
    
    # 输入梯度 dx_t = dz_t @ W_hx
    dx_t = np.dot(dz_t, W_hx)
    # 上一时刻隐状态梯度 dh_prev = dz_t @ W_hh
    dh_prev = np.dot(dz_t, W_hh)
    
    # 权重梯度
    dW_hx = np.dot(dz_t.T, x_t)  # (hidden, batch) @ (batch, input)
    dW_hh = np.dot(dz_t.T, h_prev)
    # 偏置梯度，按batch求和
    db_h = np.sum(dz_t, axis=0, keepdims=True)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# 测试代码
if __name__ == "__main__":
    # 超参
    batch = 2
    input_dim = 3
    hidden_dim = 4
    
    # 随机初始化输入与权重
    x = np.random.randn(batch, input_dim)
    h_prev = np.random.randn(batch, hidden_dim)
    W_hx = np.random.randn(hidden_dim, input_dim)
    W_hh = np.random.randn(hidden_dim, hidden_dim)
    b_h = np.random.randn(1, hidden_dim)
    
    # 前向
    h_t, cache = rnn_forward(x, h_prev, W_hx, W_hh, b_h)
    print("前向 h_t shape:", h_t.shape)
    
    # 模拟上游梯度
    dh_next = np.random.randn(batch, hidden_dim)
    dx, dh_prev_grad, dWhx, dWhh, dbh = rnn_backward(dh_next, cache)
    
    print("dx_t shape:", dx.shape)
    print("dh_prev shape:", dh_prev_grad.shape)
    print("dW_hx shape:", dWhx.shape)
    print("dW_hh shape:", dWhh.shape)
    print("db_h shape:", dbh.shape)

前向 h_t shape: (2, 4)
dx_t shape: (2, 3)
dh_prev shape: (2, 4)
dW_hx shape: (4, 3)
dW_hh shape: (4, 4)
db_h shape: (1, 4)


# 4 高级循环网络 4.1 计算题
## 已知条件
深度双向RNN：L层，每层单向隐藏单元H，输入维度D，输出维度O；包含权重+偏置，忽略嵌入、中间投影，仅算循环层+最终输出层参数。

## 一、单层单向RNN参数（单方向）
1. 输入权重：$W_{xh} \in \mathbb{R}^{H\times in}$
2. 循环权重：$W_{hh} \in \mathbb{R}^{H\times H}$
3. 偏置：$b_h \in \mathbb{R}^H$

### 第1层单向（输入是D维）
参数数量：$H\cdot D + H\cdot H + H$

### 第2~L层单向（输入是上一层双向拼接2H）
参数数量：$H\cdot 2H + H\cdot H + H = 3H^2+H$

## 二、单层双向RNN参数（前向+后向两个单向）
- 第1层双向：$2\times(HD+H^2+H)$
- 第2~L层双向：$2\times(3H^2+H)$

## 三、L层双向RNN总循环层参数
$$
\begin{align*}
P_{rnn} &= 2(HD+H^2+H) + (L-1)\cdot 2(3H^2+H) \\
&= 2HD + 2H + 2H^2 + 6(L-1)H^2 + 2(L-1)H \\
&= 2HD + 2LH + \big[2+6(L-1)\big]H^2 \\
&= 2HD + 2LH + (6L-4)H^2
\end{align*}
$$

## 四、输出层参数
双向最后一层输出拼接维度 $2H$
权重：$O\cdot 2H$，偏置：$O$
$$P_{out}=2HO + O$$

## 五、模型总参数
$$
\begin{align*}
P_{total} &= P_{rnn} + P_{out} \\
&= \boldsymbol{2HD + (6L-4)H^2 + 2LH + 2HO + O}
\end{align*}
$$
```

In [3]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # 双向单层RNN，batch_first=False 匹配输入形状(seq_len, batch, input_dim)
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            bidirectional=True,
            num_layers=1,
            batch_first=False
        )
        self.hidden_dim = hidden_dim

    def forward(self, X):
        """
        参数：
            X: 输入序列，shape (seq_len, batch, input_dim)
        返回：
            concat_seq_hidden: 每个时间步拼接隐状态 (seq_len, batch, 2*hidden_dim)
            seq_repr: 全局序列表示，最后一步拼接隐状态 (batch, 2*hidden_dim)
        """
        # out: (seq_len, batch, 2*hidden_dim) 每一步前向+后向拼接
        # hn: (2, batch, hidden_dim)，0行前向最后隐状态，1行后向最后隐状态
        out, hn = self.rnn(X)
        
        # 全局序列表示：拼接前向最后一步、后向最后一步
        forward_last = hn[0]   # (batch, hidden_dim)
        backward_last = hn[1]  # (batch, hidden_dim)
        seq_repr = torch.cat([forward_last, backward_last], dim=-1) # (batch, 2*hidden_dim)
        
        return out, seq_repr

# 测试代码
if __name__ == "__main__":
    # 超参
    seq_len = 5
    batch = 3
    input_dim = 4
    hidden_dim = 6
    
    # 初始化模型与输入
    model = BiRNNEncoder(input_dim, hidden_dim)
    X = torch.randn(seq_len, batch, input_dim)
    
    # 前向传播
    concat_seq_h, seq_global = model(X)
    print("逐时间步拼接隐状态 shape:", concat_seq_h.shape) # (5,3,12)
    print("全局序列表示 shape:", seq_global.shape)          # (3,12)

逐时间步拼接隐状态 shape: torch.Size([5, 3, 12])
全局序列表示 shape: torch.Size([3, 12])


# 5 嵌入向量 5.1 计算题
## 已知符号
- $v_c$：中心词 $w_c$ 输入词向量
- $u_o$：正例上下文词 $w_o$ 输出向量
- $u_{n_k}$：第 $k$ 个负样本输出向量，共 $K$ 个负样本
- $\sigma(x) = \dfrac{1}{1+e^{-x}}$：sigmoid激活函数

## 一、单样本对数似然目标函数推导
1. 正样本：最大化中心词匹配上下文的概率 $\sigma(v_c^\top u_o)$
2. 负样本：最大化中心词不匹配负样本的概率 $\sigma(-v_c^\top u_{n_k})$
取负对数似然作为损失，单组 $(w_c,w_o)$ 损失：
$$
L = -\Big[\log\sigma(v_c^\top u_o) + \sum_{k=1}^K \log\sigma(-v_c^\top u_{n_k})\Big]
$$

## 二、完整全局目标函数
遍历全部中心词-上下文样本对，总损失：
$$
\mathcal{L} = -\sum_{(w_c,w_o)} \left[ \log\sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_o) + \sum_{k=1}^K \log\sigma(-\boldsymbol{v}_c^\top \boldsymbol{u}_{n_k}) \right]
$$

## 三、负样本采样规则（噪声分布）
1. 采用词频3/4次幂分布 $P(w) = \dfrac{\text{count}(w)^{3/4}}{\sum_{w'}\text{count}(w')^{3/4}}$
2. 流程：
   - 根据词频噪声分布随机抽取 $K$ 个词
   - 过滤掉与正例 $w_o$ 重复的词，保证负样本不与真实上下文重合

In [4]:
import torch
import torch.nn.functional as F

def cbow_forward_loss(context_batch, target_batch, W, W_out):
    """
    CBOW 前向传播 + 完整softmax交叉熵损失计算
    参数说明：
        context_batch: 批次上下文索引，shape [batch_size, context_size]
        target_batch: 批次中心词索引，shape [batch_size]
        W: 输入嵌入矩阵 [V, d]
        W_out: 输出权重矩阵 [d, V]
    返回：
        loss: 批次平均交叉熵损失标量
    """
    batch_size, context_size = context_batch.shape
    
    # 1. 取出所有上下文词嵌入 [batch, context_size, d]
    context_embeds = W[context_batch]
    
    # 2. 求平均上下文向量（隐藏层）[batch, d]
    hidden = torch.mean(context_embeds, dim=1)
    
    # 3. 计算得分 logits [batch, V]
    logits = torch.matmul(hidden, W_out)
    
    # 4. 完整softmax + 交叉熵损失
    loss = F.cross_entropy(logits, target_batch)
    return loss

# 测试示例
if __name__ == "__main__":
    # 超参
    V = 10  # 词汇量
    d = 4   # 嵌入维度
    batch_size = 2
    context_size = 2
    
    # 构造输入
    context_batch = torch.tensor([[1,3], [2,5]])  # 2个样本，各2个上下文词
    target_batch = torch.tensor([2, 4])           # 对应中心词索引
    
    # 初始化权重
    W = torch.randn(V, d)
    W_out = torch.randn(d, V)
    
    loss_val = cbow_forward_loss(context_batch, target_batch, W, W_out)
    print(f"CBOW 批次损失值: {loss_val.item():.4f}")

CBOW 批次损失值: 2.1816


# 6 注意力机制 6.1 计算题
## 已知条件
- $Q\in\mathbb{R}^{2\times4},\ K\in\mathbb{R}^{3\times4},\ V\in\mathbb{R}^{3\times5}$
- 缩放因子 $d_k=4,\ \sqrt{d_k}=2$
- 缩放点积公式：$\text{Attention}(Q,K,V)=\text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$
- 无掩码

## 步骤1：计算原始点积 $QK^\top$
$Q$：$2\times4$，$K^\top$：$4\times3$
矩阵乘积维度：$\boldsymbol{2\times3}$

## 步骤2：缩放得分矩阵 $\text{score}=\dfrac{QK^\top}{\sqrt{d_k}}=\dfrac{QK^\top}{2}$
得分矩阵维度：$\boldsymbol{2\times3}$

## 步骤3：对得分矩阵逐行做softmax，得到注意力权重矩阵$A$
每行3个元素归一化，$\sum_{j=1}^3 A_{i,j}=1$
权重矩阵维度：$\boldsymbol{2\times3}$

## 步骤4：加权求和得到输出
$\text{Output}=A \cdot V$
$A:2\times3,\ V:3\times5$
输出矩阵维度：$\boldsymbol{2\times5}$

## 完整流程公式
$$
\begin{align*}
\text{Score} &= \frac{QK^\top}{\sqrt{4}} = \frac{1}{2}QK^\top \quad (2\times3)\\
A &= \text{softmax}(\text{Score}) \quad (2\times3)\\
\text{Output} &= A V \quad (2\times5)
\end{align*}
$$

## 最终结论
缩放点积注意力输出矩阵形状为 $\boldsymbol{\mathbb{R}^{2\times5}}$

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        # 题目固定超参
        self.num_heads = 2
        self.d_model = 4
        self.d_k = self.d_model // self.num_heads  # d_k=2
        
        # Q/K/V 投影层，输出维度 d_model
        self.w_q = nn.Linear(self.d_model, self.d_model)
        self.w_k = nn.Linear(self.d_model, self.d_model)
        self.w_v = nn.Linear(self.d_model, self.d_model)
        # 多头拼接后最终线性层
        self.w_o = nn.Linear(self.d_model, self.d_model)

    def scaled_dot_product_attn(self, q, k, v):
        """单头缩放点积注意力"""
        attn_score = torch.matmul(q, k.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))
        attn_weight = F.softmax(attn_score, dim=-1)
        out = torch.matmul(attn_weight, v)
        return out

    def forward(self, X):
        """
        X: (seq_len, batch, d_model)
        return out: (seq_len, batch, d_model) 与输入同形状
        """
        seq_len, batch, _ = X.shape
        
        # 1. 线性投影 Q,K,V
        Q = self.w_q(X)  # (seq_len, batch, 4)
        K = self.w_k(X)
        V = self.w_v(X)
        
        # 2. 分头: (seq_len, batch, num_heads, d_k)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k).transpose(1,2)  # (seq_len, heads, batch, d_k)
        K = K.view(seq_len, batch, self.num_heads, self.d_k).transpose(1,2)
        V = V.view(seq_len, batch, self.num_heads, self.d_k).transpose(1,2)
        
        # 3. 每个头计算缩放点积注意力
        attn_out = self.scaled_dot_product_attn(Q, K, V)  # (seq_len, heads, batch, d_k)
        
        # 4. 拼接所有头
        attn_out = attn_out.transpose(1,2).contiguous()  # (seq_len, batch, heads, d_k)
        concat = attn_out.view(seq_len, batch, self.d_model)  # heads*d_k = d_model
        
        # 5. 最终线性层输出
        output = self.w_o(concat)
        return output

# 测试代码
if __name__ == "__main__":
    mha = MultiHeadAttention()
    seq_len, batch = 5, 3
    X = torch.randn(seq_len, batch, 4)  # (seq_len, batch, d_model=4)
    res = mha(X)
    print("输入形状:", X.shape)
    print("输出形状:", res.shape)

输入形状: torch.Size([5, 3, 4])
输出形状: torch.Size([5, 3, 4])
